# 01 Manipulation
Codecademy: Intro to SQL, lesson 1. SQLite, run in Jupyter with JupySQL.

Own dataset instead of Codecademy's `celebs`: a small portfolio of Vienna-listed stocks (made-up prices).

Covers: `CREATE TABLE`, constraints, `INSERT`, `SELECT`, `ALTER TABLE`, `UPDATE`, `DELETE`.

In [24]:
# turns on the %%sql cell magic
%load_ext sql
# opens data\portfolio.db, creates the file if it doesn't exist
%sql sqlite:///../data/portfolio.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 1. Basics

- **SQL**: language for **relational databases**, which store data in **tables** (also called *relations*)
- **Row** = one record, **column** = one field with one data type. Like a DataFrame that lives in a database
- **Statement** = one instruction, ends with `;`. Line breaks and spaces don't matter
- **Clauses** (`SELECT`, `FROM`, ...) are written in CAPS by convention; SQL itself is case-insensitive

Anatomy:

    CREATE TABLE  holdings  (ticker TEXT, shares INTEGER);
    ------------  --------  -----------------------------
       clause      table    parameter: columns + data types

| Data type | Holds | Example |
|---|---|---|
| `INTEGER` | whole number | `50` (shares) |
| `REAL` | decimal number | `62.40` (price) |
| `TEXT` | text, in single quotes | `'EBS'` |
| `DATE` | date as `YYYY-MM-DD` | `'2026-11-06'` |

SQLite has no real `DATE` (or `UUID`) type: both are stored as `TEXT`, so write dates exactly as `YYYY-MM-DD` to sort correctly.

## 2. Create

`CREATE TABLE` makes a new, empty table. **Constraints** after the data type are rules the database enforces:

| Constraint | Rule |
|---|---|
| `PRIMARY KEY` | unique row ID, only 1 per table |
| `UNIQUE` | no duplicate values (many per table allowed) |
| `NOT NULL` | value required |
| `DEFAULT x` | `x` is used when no value is given |

In [25]:
%%sql
-- DROP TABLE IF EXISTS deletes the table if it's there, so the notebook can be re-run
DROP TABLE IF EXISTS holdings;

CREATE TABLE holdings (
   id INTEGER PRIMARY KEY,
   ticker TEXT UNIQUE NOT NULL,
   company TEXT,
   shares INTEGER NOT NULL,
   buy_price REAL,
   currency TEXT DEFAULT 'EUR'
);

Running query in 'sqlite:///../data/portfolio.db'

++
||
++
++

## 3. Insert

`INSERT INTO` adds rows. Values go in the same order as the listed columns; several rows are separated by commas.
Columns left out get their `DEFAULT` (here `currency` = `'EUR'`) or `NULL`.

In [26]:
%%sql
INSERT INTO holdings (id, ticker, company, shares, buy_price)
VALUES (1, 'EBS',  'Erste Group',            50, 62.40),
       (2, 'OMV',  'OMV',                    80, 45.10),
       (3, 'VER',  'Verbund',                30, 68.90),
       (4, 'ANDR', 'Andritz',                40, 58.20),
       (5, 'VIG',  'Vienna Insurance Group', 100, 30.50);

Running query in 'sqlite:///../data/portfolio.db'

5 rows affected.

++
||
++
++

## 4. Select

`SELECT` fetches data and returns a new table, the **result set**. `*` = all columns.

| SQL | pandas |
|---|---|
| `SELECT * FROM holdings;` | `df` |
| `SELECT ticker, shares FROM holdings;` | `df[['ticker', 'shares']]` |

In [27]:
%%sql
SELECT ticker, shares, buy_price, currency FROM holdings;

Running query in 'sqlite:///../data/portfolio.db'

ticker,shares,buy_price,currency
EBS,50,62.4,EUR
OMV,80,45.1,EUR
VER,30,68.9,EUR
ANDR,40,58.2,EUR
VIG,100,30.5,EUR


## 5. Alter

`ALTER TABLE ... ADD COLUMN` adds a column. Existing rows get `NULL` there (= missing, no value).

pandas: `df['sector'] = None`

In [28]:
%%sql
ALTER TABLE holdings
ADD COLUMN sector TEXT;

SELECT * FROM holdings;

Running query in 'sqlite:///../data/portfolio.db'

id,ticker,company,shares,buy_price,currency,sector
1,EBS,Erste Group,50,62.4,EUR,None
2,OMV,OMV,80,45.1,EUR,None
3,VER,Verbund,30,68.9,EUR,None
4,ANDR,Andritz,40,58.2,EUR,None
5,VIG,Vienna Insurance Group,100,30.5,EUR,None


## 6. Update

`UPDATE ... SET ... WHERE` edits existing rows. **Without `WHERE`, every row is changed.**

pandas: `df.loc[df['ticker'] == 'EBS', 'sector'] = 'Financials'`

Only 3 positions get a sector; `ANDR` and `VIG` stay `NULL`.

In [29]:
%%sql
UPDATE holdings SET sector = 'Financials' WHERE ticker = 'EBS';
UPDATE holdings SET sector = 'Energy'     WHERE ticker = 'OMV';
UPDATE holdings SET sector = 'Utilities'  WHERE ticker = 'VER';

SELECT * FROM holdings;

Running query in 'sqlite:///../data/portfolio.db'

1 rows affected.

1 rows affected.

1 rows affected.

id,ticker,company,shares,buy_price,currency,sector
1,EBS,Erste Group,50,62.4,EUR,Financials
2,OMV,OMV,80,45.1,EUR,Energy
3,VER,Verbund,30,68.9,EUR,Utilities
4,ANDR,Andritz,40,58.2,EUR,None
5,VIG,Vienna Insurance Group,100,30.5,EUR,None


## 7. Delete

`DELETE FROM ... WHERE` removes rows. **Without `WHERE`, every row is deleted.**

Sell all positions without a sector:
- `IS NULL` / `IS NOT NULL` test for missing values
- Pitfall: `= NULL` never matches anything, because `NULL` isn't a value

In [30]:
%%sql
DELETE FROM holdings
WHERE sector IS NULL;

SELECT * FROM holdings;   -- 3 rows left: EBS, OMV, VER

Running query in 'sqlite:///../data/portfolio.db'

2 rows affected.

id,ticker,company,shares,buy_price,currency,sector
1,EBS,Erste Group,50,62.4,EUR,Financials
2,OMV,OMV,80,45.1,EUR,Energy
3,VER,Verbund,30,68.9,EUR,Utilities


## 8. Constraint errors

Inserts that break the constraints from section 2 are rejected. Not run here, an error would stop Run All:

    INSERT INTO holdings (id, ticker, shares) VALUES (1, 'POST', 20);
    -> UNIQUE constraint failed: holdings.id         (PRIMARY KEY 1 already exists)

    INSERT INTO holdings (id, ticker, shares) VALUES (6, 'EBS', 20);
    -> UNIQUE constraint failed: holdings.ticker     (EBS already exists)

    INSERT INTO holdings (id, ticker) VALUES (6, 'POST');
    -> NOT NULL constraint failed: holdings.shares   (shares is required)

Codecademy shows nothing when an insert fails; Jupyter shows the error.